# run_capture: replay recorded games on the board and save labeled crops (M2)

Needs the real camera (or Gazebo frames). Not executed in CI. For each game in
`data/games/*.pgn` the human sets up each position in turn; every frame is warped, cut into
64 crops and labeled from the PGN (`capture.ipynb`). Output: `data/crops/<session_id>/`.

**TODO**
- [ ] implement `capture_game` in `capture.ipynb`
- [ ] one session per lighting setup / camera placement; name sim sessions `sim_*`, real `sNN_*`
- [ ] check a few crops by eye after each session (cell 3)

In [ ]:
%run ./calibration.ipynb
%run ./capture.ipynb
%cd ..
from pathlib import Path

import cv2

cam_cfg = load_config("configs/camera.yaml")
geom = load_board_geometry()
intr = load_intrinsics() if default_intrinsics_path().exists() else None
SESSION_ID = "s01_daylight"           # TODO: one per lighting / camera setup
CROPS_ROOT = load_config("configs/train.yaml").data.crops_root
GAMES = sorted(Path("data/games").glob("*.pgn"))
print(len(GAMES), "games")

## Capture

In [ ]:
writer = ManifestWriter(CROPS_ROOT, SESSION_ID)
cap = cv2.VideoCapture(cam_cfg.camera.device)
try:
    for pgn in GAMES:
        n = capture_game(cap, pgn, writer, pgn.stem, geom, cam_cfg, intr)   # TODO M2
        print(pgn.stem, n, "frames")
finally:
    cap.release()

## Inspect

In [ ]:
import matplotlib.pyplot as plt

df = load_manifest(CROPS_ROOT, [SESSION_ID])
print(df.label.value_counts())
sample = df[df.label != "empty"].sample(8, random_state=0)
fig, axes = plt.subplots(1, 8, figsize=(16, 4))
for ax, (_, row) in zip(axes, sample.iterrows()):
    ax.imshow(read_crop(row.abs_path)); ax.set_title(f"{row.square} {row.label}"); ax.axis("off")